# Budget GPU Memory for the KV Cache

Companion notebook for the [Memory Budgeting](https://codemia.io/courses/llm_inference_and_serving/llm_memory_budgeting) lesson on Codemia.

**What you will build**: the memory plan a serving engineer writes before choosing a GPU. You will compute the KV cache size per token from a model config, subtract weights and overheads from GPU memory to get the cache budget, turn it into a token capacity and a number of concurrent sequences, and then replace the average sequence length with a realistic length distribution to see how much a few long requests change the answer.

No GPU. Standard library, `numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
GB = 1e9
KiB = 1024

## 1. Cache bytes per token

Every layer stores a key and a value vector for every key value head:

`bytes per token = 2 * layers * kv_heads * head_dim * bytes_per_value`

Grouped query attention shares each key value head across several query heads, which is why Llama 3 8B needs a quarter of the cache of Llama 2 7B.

In [ ]:
MODELS = {
    #                 params(B) layers kv_heads head_dim
    "Llama 2 7B":     (6.7,  32, 32, 128),
    "Llama 3 8B":     (8.0,  32,  8, 128),
    "Llama 3 70B":    (70.6, 80,  8, 128),
    "Mistral 7B":     (7.2,  32,  8, 128),
}

def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_value=2):
    return 2 * layers * kv_heads * head_dim * bytes_per_value

for name, (p, L, H, D) in MODELS.items():
    print("%-12s %5.0f KiB per token in 16 bit" % (name, kv_bytes_per_token(L, H, D) / KiB))

## 2. From GPU memory to a cache budget

Engines reserve a fraction of GPU memory, for example vLLM's default utilization of 0.9, and give what is left after weights, runtime overhead and activation workspace to the cache. Tensor parallelism splits both the weights and the cache across GPUs.

In [ ]:
def cache_budget(model, gpu_gb=80, gpus=1, weight_bytes=2, util=0.9, overhead_gb=3, activation_gb=2):
    p, L, H, D = MODELS[model]
    weights_per_gpu = p * 1e9 * weight_bytes / gpus
    per_gpu = gpu_gb * GB * util - weights_per_gpu - (overhead_gb + activation_gb) * GB
    return max(per_gpu, 0) * gpus

def token_capacity(model, kv_value_bytes=2, **kw):
    p, L, H, D = MODELS[model]
    return int(cache_budget(model, **kw) // kv_bytes_per_token(L, H, D, kv_value_bytes))

print("Llama 3 8B on one 80 GB GPU:    %8d tokens" % token_capacity("Llama 3 8B"))
print("Llama 3 8B on one 24 GB GPU:    %8d tokens" % token_capacity("Llama 3 8B", gpu_gb=24))
print("  ... with 4 bit weights:       %8d tokens" % token_capacity("Llama 3 8B", gpu_gb=24, weight_bytes=0.5))
print("Llama 3 70B on 4 x 80 GB:       %8d tokens" % token_capacity("Llama 3 70B", gpus=4))
print("Llama 2 7B on one 80 GB GPU:    %8d tokens" % token_capacity("Llama 2 7B"))

## 3. Concurrency versus context length

Dividing token capacity by the active length of a sequence gives the number of sequences that fit. Plot it for each deployment as context grows. Long context products pay for it in concurrency, and therefore in cost per token.

In [ ]:
ctx = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])
setups = [("Llama 3 8B, 1 x 80 GB", dict(model="Llama 3 8B")),
          ("Llama 3 8B, 1 x 80 GB, 8 bit cache", dict(model="Llama 3 8B", kv_value_bytes=1)),
          ("Llama 2 7B, 1 x 80 GB", dict(model="Llama 2 7B")),
          ("Llama 3 70B, 4 x 80 GB", dict(model="Llama 3 70B", gpus=4))]
plt.figure(figsize=(7.5, 4))
for label, kw in setups:
    cap = token_capacity(**kw)
    plt.plot(ctx, np.maximum(cap // ctx, 0), marker="o", label=label)
plt.xscale("log", base=2); plt.yscale("log")
plt.xlabel("active tokens per sequence"); plt.ylabel("concurrent sequences that fit")
plt.legend(fontsize=8); plt.grid(alpha=0.3, which="both")
plt.title("Every doubling of context halves concurrency")
plt.show()

## 4. Averages lie: a realistic length distribution

Real traffic mixes short chats with a few huge documents. Sample sequence lengths from a lognormal with a median around 1,500 tokens and a long tail, then admit them first come, first served into a paged cache with 16 token blocks. Compare the count with the naive estimate of capacity divided by the mean.

In [ ]:
def sample_lengths(n, median=1500, sigma=1.0, cap=131072):
    return np.minimum(rng.lognormal(np.log(median), sigma, n), cap).astype(int) + 1

def admit(capacity_tokens, lengths, block=16, headroom=0.0):
    usable = int(capacity_tokens // block * (1 - headroom))
    need = -(-lengths // block)
    fits = np.cumsum(need) <= usable
    return int(np.argmin(fits)) if not fits.all() else len(lengths)

cap = token_capacity("Llama 3 8B")
trials = [admit(cap, sample_lengths(5000)) for _ in range(300)]
naive = cap / sample_lengths(200_000).mean()
print("capacity %d tokens" % cap)
print("naive estimate (capacity / mean length): %.0f sequences" % naive)
print("simulated admission: median %d, 5th percentile %d" % (np.median(trials), np.percentile(trials, 5)))
plt.figure(figsize=(6, 3))
plt.hist(trials, bins=30); plt.axvline(naive, color="black", ls="--", label="capacity / mean")
plt.xlabel("sequences admitted"); plt.legend(); plt.title("A few long sequences decide the batch")
plt.show()

The median batch lands near the naive estimate, but the spread is wide and the low tail is what sets your latency and throughput guarantees on a bad minute. That is why engines admit by blocks free rather than by a fixed batch size, and why capacity plans should use a percentile of the distribution.

## 5. Exercises

1. Add a 4,000 token system prompt to every request and assume prefix caching stores it once. How many more sequences fit?
2. Find the smallest GPU memory size on which Llama 3 70B in 8 bit weights on two GPUs still holds 64 sequences of 8,000 tokens.
3. Add a headroom of 15 percent to `admit`. How does it shift the 5th percentile, and what does that headroom buy during decode?